In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# Euler resold vehicles — resale-date cohorts (>=3 vehicles), last-point outliers removed, XGB prediction to resale

Cohort key = resale date; only resale months with **>=3 well-observed** (>=12 months of SoH before resale, data ending within 6 months of resale) usable-SoH Euler vehicles are plotted (currently
June 2025: 217094, 217395, 217380). Two are short-history repossessed Hilolds whose BMS SoH ends in a
**terminal one-month cliff** (217380: 91->66, 217395: 83->69) — a remaining-capacity glitch, not wear.
We **drop that last outlier point** (any >10 pp single-step SoH drop) and **predict forward with the XGB
rate model** (`euler_model.free_run`) from the clean trajectory through the resale date, so SoH-at-resale is
the model **prediction**. solid = measured, dashed o = prediction (to the resale date), star = SoH at resale,
orange line = resale date, triangle = projected SoH at the 5-yr warranty end.

In [ ]:
import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.express as px
import euler_features, euler_model as em, config
PAL = px.colors.qualitative.Dark24
INHOUSE_YRS = 2     # in-house (Turno) warranty, runs from the resale date
MIN_BEFORE = 6      # min months of SoH observed BEFORE resale
MAX_GAP = 18        # max months between the last reading and the resale date
TODAY = pd.Timestamp('2026-06-19')

def clean_g(g):
    # drop capacity-glitch outliers: a month whose SoH falls >10 pp below the last kept month
    g = g.sort_values('month'); rows = []; last = None
    for _, r in g.iterrows():
        if last is not None and (last - r['soh']) > 10.0:
            continue
        rows.append(r); last = r['soh']
    return pd.DataFrame(rows)

if not Path('data/euler/features/feature_table.parquet').exists():
    euler_features.main()
meu_raw = pd.read_parquet('data/euler/features/feature_table.parquet').sort_values(['vin', 'month'])
meu = pd.concat([clean_g(g) for _, g in meu_raw.groupby('vin')], ignore_index=True).sort_values(['vin', 'month'])
emodel = em.train(em.build_transitions(meu))

rs = pd.read_csv('data/Repo - Pricing - Sold.csv'); rs.columns = [c.strip() for c in rs.columns]
rs = rs[rs['OEM'].astype(str).str.contains('euler', case=False, na=False)].copy()
rs['reg'] = pd.to_datetime(rs['Registration Date'], format='%d-%b-%Y', errors='coerce')
rs['resale'] = pd.to_datetime(rs['Resale Date'], format='%d-%b-%Y', errors='coerce')
rs['odo'] = pd.to_numeric(rs['Odometer reading (current)'], errors='coerce')
have = set(meu['vin'].unique())
rs = rs[rs['VIN'].isin(have) & rs['resale'].notna()].copy()
rs['resmonth'] = rs['resale'].dt.to_period('M')

def resold_d(r):
    vin = r['VIN']; g = meu[meu['vin'] == vin].sort_values('month'); reg = r['reg']
    cal_obs = list(g['month']); s_obs = g['soh'].to_numpy(); last = g['month'].iloc[-1]
    n_before = int((g['month'] <= r['resale']).sum()); gap = max(0, (r['resale'].to_period('M') - last.to_period('M')).n)
    inhouse_end = r['resale'] + pd.DateOffset(years=INHOUSE_YRS)
    last_age = float(g['age_months'].iloc[-1])
    Hmax = max(int(round(8 * 12 - last_age)), (inhouse_end.to_period('M') - last.to_period('M')).n, 1)
    fcf = np.clip(np.array(em.free_run(g, emodel, Hmax)), 0, 100)                 # full prediction
    cut = (int(np.argmax(fcf <= 80)) + 1) if (fcf <= 80).any() else len(fcf)      # display only down to 80% EoFL
    fc = fcf[:cut]
    cal_fc = [last + pd.DateOffset(months=k) for k in range(1, len(fc) + 1)]
    cal_full = [last + pd.DateOffset(months=k) for k in range(1, len(fcf) + 1)]
    ax_obs = (g['month'] - reg).dt.days.to_numpy()
    if r['resale'] <= last:
        soh_resale = float(np.interp((r['resale'] - reg).days, ax_obs, s_obs)); predicted = False
    else:
        fxf = np.array([(m - reg).days for m in cal_full]); soh_resale = float(np.interp((r['resale'] - reg).days, fxf, fcf)); predicted = True
    return dict(vin=vin, tail=vin[-6:], model=str(r['Model']), reg=reg, resale=r['resale'], odo=r['odo'],
                cal_obs=cal_obs, s_obs=s_obs, cal_fc=cal_fc, s_fc=fc, soh_resale=soh_resale, predicted=predicted,
                inhouse_end=inhouse_end, soh_now=float(s_obs[-1]), n_before=n_before, gap=gap, wyr=5)

print('usable-SoH resold Euler by resale month:')
for m, grp in rs.groupby('resmonth'):
    print(f'  {m}: {len(grp)} -> ' + str([(v[-6:], str(md)[:8]) for v, md in zip(grp['VIN'], grp['Model'])]))

## Resale months with >=3 vehicles — last-point outliers removed, XGB prediction carried to the resale date

In [ ]:
def show_resale_month(period, exclude=()):
    cohort = [resold_d(r) for _, r in rs[rs['resmonth'] == period].iterrows()]
    cohort = [d for d in cohort if d['tail'] not in exclude and d['vin'] not in exclude]
    cohort = [d for d in cohort if d['n_before'] >= MIN_BEFORE and d['gap'] <= MAX_GAP]   # keep only well-observed-before-resale vehicles
    cohort = [d for d in cohort if d['soh_resale'] < 99.5]   # drop flat vehicles at 100% SoH at resale (no degradation signal)
    if len(cohort) < 3:
        return
    cohort = sorted(cohort, key=lambda d: -d['soh_resale'])
    resale_date = cohort[0]['resale']; inhouse_end = cohort[0]['inhouse_end']
    print(f'resold {period}: {len(cohort)} -> ' + str([(d['tail'], f"{d['soh_resale']:.0f}%" + ('(pred)' if d['predicted'] else '(meas)'), f"{d['n_before']}mo", d['model'][:8]) for d in cohort]))
    ymin = 78
    fig = go.Figure()
    fig.add_scatter(x=[resale_date, inhouse_end, inhouse_end, resale_date], y=[ymin, ymin, 101, 101], fill='toself',
                    fillcolor='rgba(0,150,136,0.08)', line=dict(width=0), mode='lines', showlegend=False, hoverinfo='skip')
    for i, d in enumerate(cohort):
        col = PAL[i % len(PAL)]; tail = d['tail']; good = d['soh_resale'] >= 80
        # 100% SoH at registration: DISPLAY-ONLY marker (never fed to the prediction)
        fig.add_scatter(x=[d['reg']], y=[100], mode='markers', legendgroup=tail, showlegend=False,
                        marker=dict(symbol='circle', size=8, color=col, line=dict(color='black', width=0.6)),
                        hovertemplate=tail + ' registered ' + pd.Timestamp(d['reg']).strftime('%b %Y') + ' (100%, display only)<extra></extra>')
        if d['s_obs'][0] >= 98:   # draw the dotted lead-in ONLY when the first reading is ~100% (accurate); green (first<100) gets the marker alone
            fig.add_scatter(x=[d['reg'], d['cal_obs'][0]], y=[100, d['s_obs'][0]], mode='lines', line=dict(color=col, width=1, dash='dot'),
                            opacity=0.5, legendgroup=tail, showlegend=False, hoverinfo='skip')
        fig.add_scatter(x=d['cal_obs'], y=list(d['s_obs']), mode='lines+markers', line=dict(color=col, width=1.8),
                        marker=dict(size=6, color=col), legendgroup=tail,
                        name=f"{tail} . {d['soh_resale']:.0f}% @ resale" + (' OK' if good else ''),
                        hovertemplate=tail + ' actual %{y:.1f}%  %{x|%b %Y}<extra></extra>')
        fig.add_scatter(x=d['cal_fc'], y=list(d['s_fc']), mode='lines+markers', line=dict(color=col, width=1.8, dash='dash'),
                        marker=dict(size=4, color=col, symbol='circle-open'), legendgroup=tail, showlegend=False,
                        hovertemplate=tail + ' prediction %{y:.1f}%  %{x|%b %Y}<extra></extra>')
        fig.add_scatter(x=[d['resale']], y=[d['soh_resale']], mode='markers', legendgroup=tail, showlegend=False,
                        marker=dict(symbol='star', size=15, color=col, line=dict(color='black', width=0.6)),
                        hovertemplate=tail + f" resale {d['soh_resale']:.0f}%<extra></extra>")
        if d['s_fc'][-1] <= 80.5:
            fig.add_scatter(x=[d['cal_fc'][-1]], y=[d['s_fc'][-1]], mode='markers+text', legendgroup=tail, showlegend=False,
                            marker=dict(symbol='circle-open', size=10, color=col, line=dict(width=2)),
                            text=[pd.Timestamp(d['cal_fc'][-1]).strftime('%b %Y')], textposition='bottom center',
                            textfont=dict(size=8, color=col), hovertemplate=tail + ' hits 80%%<extra></extra>')
    fig.add_hline(y=80, line=dict(color='red', width=1.3, dash='dot'), annotation_text='80% EoFL', annotation_position='bottom left')
    fig.add_scatter(x=[resale_date, resale_date], y=[ymin, 101], mode='lines', line=dict(color='darkorange', width=1.6, dash='dashdot'),
                    showlegend=False, hoverinfo='skip')
    fig.add_annotation(x=resale_date, y=101, text=f'resold {period}', showarrow=False, yanchor='bottom', font=dict(color='darkorange', size=11))
    fig.add_scatter(x=[inhouse_end, inhouse_end], y=[ymin, 101], mode='lines', line=dict(color='teal', width=1.6, dash='dash'),
                    showlegend=False, hoverinfo='skip')
    fig.add_annotation(x=inhouse_end, y=101, text='2-yr in-house warranty ends ' + pd.Timestamp(inhouse_end).strftime('%b %Y'),
                       showarrow=False, yanchor='bottom', xanchor='right', font=dict(color='teal', size=10))
    ref = pd.Timestamp(int(np.median([pd.Timestamp(d['reg']).value for d in cohort if pd.notna(d.get('reg'))])))
    _allx = [m for d in cohort for m in (list(d['cal_obs']) + list(d['cal_fc']))]
    xmin = min(_allx); xmax = max(_allx + [inhouse_end])
    fig.add_scatter(x=[TODAY, TODAY], y=[ymin, 101], mode='lines', line=dict(color='black', width=1.4, dash='dashdot'), showlegend=False, hoverinfo='skip')
    fig.add_annotation(x=TODAY, y=ymin, text=TODAY.strftime('%d %b %Y'), showarrow=False, yanchor='bottom', xanchor='left', font=dict(color='black', size=10))
    yrt = pd.date_range(pd.Timestamp(xmin).to_period('Y').to_timestamp(), pd.Timestamp(xmax) + pd.Timedelta(days=370), freq='YS')
    age_txt = [f"{(t - ref).days / 365.25:.0f}" for t in yrt]
    fig.add_scatter(x=[xmin], y=[ymin], xaxis='x2', mode='markers', marker=dict(opacity=0), showlegend=False, hoverinfo='skip')
    fig.update_layout(height=560, width=1150, template='plotly_white', margin=dict(t=70),
                      title=dict(text=f"Euler resold {period} (>=3 vehicles, >={MIN_BEFORE}mo SoH before resale): 100% at registration, prediction to 80% EoFL; shaded = 2-yr in-house warranty", y=0.97),
                      xaxis=dict(title='calendar (year-month)', type='date', range=[xmin, xmax]),
                      xaxis2=dict(overlaying='x', side='top', type='date', range=[xmin, xmax], tickmode='array',
                                  tickvals=list(yrt), ticktext=age_txt, title=dict(text='age since registration (yr)', standoff=6),
                                  tickfont=dict(size=9), title_font=dict(size=10)),
                      yaxis=dict(title='BMS-capacity SoH (%)', range=[ymin, 101]),
                      legend=dict(title='vehicle . SoH @ resale', x=1.01, y=1, font=dict(size=10)))
    fig.show()

cohorts = sorted(rs['resmonth'].unique())
print(f'plotting resale months with >=3 well-observed vehicles (>={MIN_BEFORE}mo SoH before resale, <={MAX_GAP}mo gap to resale):')
for p in cohorts:
    show_resale_month(p)

In [ ]:
# === Smoothed actual curves (5-pt moving average) for the resold cohorts ===
def smooth(y, w=5):
    y = np.asarray(y, float)
    if len(y) < 3:
        return y.copy()
    w = min(w, len(y)); w = w if w % 2 == 1 else w - 1; w = max(w, 3); pad = w // 2
    return np.convolve(np.pad(y, pad, mode='edge'), np.ones(w) / w, mode='valid')

if '_raw_resold_d' not in globals():
    _raw_resold_d = resold_d
def resold_d(r):                      # override: smooth the displayed ACTUAL SoH (prediction & markers unchanged)
    d = _raw_resold_d(r); d['s_obs'] = smooth(d['s_obs']); return d

print('smoothed actual curves:')
for p in sorted(rs['resmonth'].unique()):
    show_resale_month(p)

In [ ]:
# === Resold Apr-2026 cohort - publication-ready: PCHIP through original points, reaches 100, reg numbers ===
from scipy.interpolate import PchipInterpolator
SEL = ['217123', '217173', '217054']
COLORS = {'217123': '#1b9e77', '217173': '#d95f02', '217054': '#7570b3'}
_er = pd.read_csv('data/euler/Euler_Regd_Details.csv').drop_duplicates('vin'); _m1 = dict(zip(_er['vin'], _er['regd_number']))
_m2 = dict(zip(rs['VIN'], rs['Vehicle Reg No']))
def regno(vin): return _m1.get(vin) or _m2.get(vin) or vin[-6:]

def pchip_dense(cal, y, n=200):
    cal = [pd.Timestamp(t) for t in cal]; y = np.asarray(y, float)
    xn = np.array([(t - cal[0]).days for t in cal], float)
    keep = np.concatenate([[True], np.diff(xn) > 0]); xn, y = xn[keep], y[keep]
    if len(xn) < 2: return cal, list(y)
    dx = np.linspace(xn[0], xn[-1], n); dy = PchipInterpolator(xn, y)(dx)
    return [cal[0] + pd.Timedelta(days=float(x)) for x in dx], dy

_rd = _raw_resold_d if '_raw_resold_d' in globals() else resold_d
cohort = [_rd(r) for _, r in rs[rs['resmonth'] == pd.Period('2026-04', 'M')].iterrows()]
cohort = sorted([d for d in cohort if d['tail'] in SEL], key=lambda d: -d['soh_resale'])
resale_date = cohort[0]['resale']; inhouse_end = cohort[0]['inhouse_end']; ymin = 78
ref = min(pd.Timestamp(d['reg']) for d in cohort)

fig = go.Figure()
fig.add_scatter(x=[resale_date, inhouse_end, inhouse_end, resale_date], y=[ymin, ymin, 101, 101], fill='toself',
                fillcolor='rgba(120,120,120,0.06)', line=dict(width=0), mode='lines', showlegend=False, hoverinfo='skip')
for d in cohort:
    col = COLORS.get(d['tail'], '#444'); tail = d['tail']
    # smooth actual curve: anchor (registration, 100%) then PCHIP through every measured point -> reaches 100, points lie on it
    acal = [pd.Timestamp(d['reg'])] + list(d['cal_obs']); aval = [100.0] + list(d['s_obs'])
    cx, cy = pchip_dense(acal, aval)
    fig.add_scatter(x=cx, y=cy, mode='lines', line=dict(color=col, width=1.8), legendgroup=tail,
                    name=f"{regno(d['vin'])}  ·  {d['soh_resale']:.0f}% at resale", hovertemplate=tail + ' %{y:.1f}%<extra></extra>')
    fig.add_scatter(x=d['cal_obs'], y=list(d['s_obs']), mode='markers', legendgroup=tail, showlegend=False,
                    marker=dict(size=5, color=col, opacity=0.55, line=dict(width=0)),
                    hovertemplate=tail + '  measured %{y:.1f}%  %{x|%b %Y}<extra></extra>')
    # projection seeded from the last actual point (no break), PCHIP-smoothed
    fcal = [list(d['cal_obs'])[-1]] + list(d['cal_fc']); fval = [list(d['s_obs'])[-1]] + list(d['s_fc'])
    fx, fy = pchip_dense(fcal, fval)
    fig.add_scatter(x=fx, y=fy, mode='lines', line=dict(color=col, width=1.8, dash='dash'), opacity=0.85,
                    legendgroup=tail, showlegend=False, hovertemplate=tail + ' projection %{y:.1f}%<extra></extra>')
    fig.add_scatter(x=[d['resale']], y=[d['soh_resale']], mode='markers', legendgroup=tail, showlegend=False,
                    marker=dict(symbol='diamond', size=9, color=col, line=dict(color='white', width=1.2)),
                    hovertemplate=tail + f" resale {d['soh_resale']:.0f}%<extra></extra>")
fig.add_hline(y=80, line=dict(color='#c0392b', width=1, dash='dot'))
fig.add_annotation(x=ref, y=80, text='80% end of first life', showarrow=False, xanchor='left', yanchor='bottom', font=dict(color='#c0392b', size=10))
fig.add_scatter(x=[resale_date, resale_date], y=[ymin, 101], mode='lines', line=dict(color='#999', width=1, dash='dash'), showlegend=False, hoverinfo='skip')
fig.add_annotation(x=resale_date, y=ymin + 2, text='resold ' + resale_date.strftime('%b %Y'), showarrow=False, textangle=-90, yanchor='bottom', xanchor='right', font=dict(color='#555', size=10))
fig.add_annotation(x=inhouse_end, y=ymin + 0.4, text='2-yr in-house warranty', showarrow=False, yanchor='bottom', xanchor='right', font=dict(color='#888', size=9))
allx = [m for d in cohort for m in (list(d['cal_obs']) + list(d['cal_fc']))]
xmin = min([pd.Timestamp(d['reg']) for d in cohort] + allx); xmax = max(allx + [inhouse_end])
yrt = [t for t in pd.date_range(pd.Timestamp(xmin).to_period('Y').to_timestamp(), pd.Timestamp(xmax) + pd.Timedelta(days=370), freq='YS') if (t - ref).days >= 0]
age_txt = [f"{(t - ref).days / 365.25:.0f}" for t in yrt]
fig.add_scatter(x=[xmin], y=[ymin], xaxis='x2', mode='markers', marker=dict(opacity=0), showlegend=False, hoverinfo='skip')
fig.update_layout(template='plotly_white', height=560, width=980, margin=dict(t=120, r=40, b=60, l=72),
                  font=dict(family='Helvetica Neue, Arial, sans-serif', size=13, color='#222'),
                  title=dict(text='Vehicle State of Health — resold cohort (Apr 2026)', x=0.02, xanchor='left', y=0.97, yref='container', font=dict(size=18)),
                  xaxis=dict(title='calendar', type='date', range=[xmin, xmax], showgrid=True, gridcolor='rgba(0,0,0,0.06)', ticks='outside'),
                  xaxis2=dict(overlaying='x', side='top', type='date', range=[xmin, xmax], tickmode='array', tickvals=yrt, ticktext=age_txt,
                              title=dict(text='age since registration (years)', standoff=8), showgrid=False, ticks='outside', tickfont=dict(size=10)),
                  yaxis=dict(title='State of Health (%)', range=[ymin, 101], showgrid=True, gridcolor='rgba(0,0,0,0.06)', ticks='outside'),
                  legend=dict(title=dict(text='Vehicle · SoH at resale', font=dict(size=11)), x=1.01, y=1, font=dict(size=11), bgcolor='rgba(0,0,0,0)'))
fig.show()

In [ ]:
# === March-resale samples: 2 vehicles renamed, resale line moved to March ===
from scipy.interpolate import PchipInterpolator
SEL = ['217123', '217054']
NAMES = {'217123': 'March Resale Euler Sample Vehicle 1', '217054': 'March Resale Mahindra Sample Vehicle 1'}
COLORS = {'217123': '#1b9e77', '217054': '#7570b3'}

def pchip_dense(cal, y, n=200):
    cal = [pd.Timestamp(t) for t in cal]; y = np.asarray(y, float)
    xn = np.array([(t - cal[0]).days for t in cal], float)
    keep = np.concatenate([[True], np.diff(xn) > 0]); xn, y = xn[keep], y[keep]
    if len(xn) < 2: return cal, list(y)
    dx = np.linspace(xn[0], xn[-1], n); dy = PchipInterpolator(xn, y)(dx)
    return [cal[0] + pd.Timedelta(days=float(x)) for x in dx], dy

_rd = _raw_resold_d if '_raw_resold_d' in globals() else resold_d
cohort = [_rd(r) for _, r in rs[rs['resmonth'] == pd.Period('2026-04', 'M')].iterrows()]
cohort = sorted([d for d in cohort if d['tail'] in SEL], key=lambda d: -d['soh_resale'])
resale_date = pd.Timestamp('2026-03-01'); inhouse_end = resale_date + pd.DateOffset(years=2); ymin = 78
ref = min(pd.Timestamp(d['reg']) for d in cohort)

def soh_at(d, when):
    ax = [(pd.Timestamp(m) - pd.Timestamp(d['cal_obs'][0])).days for m in (list(d['cal_obs']) + list(d['cal_fc']))]
    ay = list(d['s_obs']) + list(d['s_fc'])
    return float(np.interp((pd.Timestamp(when) - pd.Timestamp(d['cal_obs'][0])).days, ax, ay))

fig = go.Figure()
fig.add_scatter(x=[resale_date, inhouse_end, inhouse_end, resale_date], y=[ymin, ymin, 101, 101], fill='toself',
                fillcolor='rgba(120,120,120,0.06)', line=dict(width=0), mode='lines', showlegend=False, hoverinfo='skip')
for d in cohort:
    col = COLORS.get(d['tail'], '#444'); tail = d['tail']; soh_m = soh_at(d, resale_date)
    acal = [pd.Timestamp(d['reg'])] + list(d['cal_obs']); aval = [100.0] + list(d['s_obs'])
    cx, cy = pchip_dense(acal, aval)
    fig.add_scatter(x=cx, y=cy, mode='lines', line=dict(color=col, width=1.8), legendgroup=tail,
                    name=f"{NAMES[tail]}  ·  {soh_m:.0f}% at resale", hovertemplate=NAMES[tail] + ' %{y:.1f}%<extra></extra>')
    fig.add_scatter(x=d['cal_obs'], y=list(d['s_obs']), mode='markers', legendgroup=tail, showlegend=False,
                    marker=dict(size=5, color=col, opacity=0.55, line=dict(width=0)),
                    hovertemplate=NAMES[tail] + '  measured %{y:.1f}%  %{x|%b %Y}<extra></extra>')
    fcal = [list(d['cal_obs'])[-1]] + list(d['cal_fc']); fval = [list(d['s_obs'])[-1]] + list(d['s_fc'])
    fx, fy = pchip_dense(fcal, fval)
    fig.add_scatter(x=fx, y=fy, mode='lines', line=dict(color=col, width=1.8, dash='dash'), opacity=0.85,
                    legendgroup=tail, showlegend=False, hovertemplate=NAMES[tail] + ' projection %{y:.1f}%<extra></extra>')
    fig.add_scatter(x=[resale_date], y=[soh_m], mode='markers', legendgroup=tail, showlegend=False,
                    marker=dict(symbol='diamond', size=9, color=col, line=dict(color='white', width=1.2)),
                    hovertemplate=NAMES[tail] + f" resale {soh_m:.0f}%<extra></extra>")
fig.add_hline(y=80, line=dict(color='#c0392b', width=1, dash='dot'))
fig.add_annotation(x=ref, y=80, text='80% end of first life', showarrow=False, xanchor='left', yanchor='bottom', font=dict(color='#c0392b', size=10))
fig.add_scatter(x=[resale_date, resale_date], y=[ymin, 101], mode='lines', line=dict(color='#999', width=1, dash='dash'), showlegend=False, hoverinfo='skip')
fig.add_annotation(x=resale_date, y=ymin + 2, text='resold ' + resale_date.strftime('%b %Y'), showarrow=False, textangle=-90, yanchor='bottom', xanchor='right', font=dict(color='#555', size=10))
fig.add_annotation(x=inhouse_end, y=ymin + 0.4, text='2-yr in-house warranty', showarrow=False, yanchor='bottom', xanchor='right', font=dict(color='#888', size=9))
allx = [m for d in cohort for m in (list(d['cal_obs']) + list(d['cal_fc']))]
xmin = min([pd.Timestamp(d['reg']) for d in cohort] + allx); xmax = max(allx + [inhouse_end])
yrt = [t for t in pd.date_range(pd.Timestamp(xmin).to_period('Y').to_timestamp(), pd.Timestamp(xmax) + pd.Timedelta(days=370), freq='YS') if (t - ref).days >= 0]
age_txt = [f"{(t - ref).days / 365.25:.0f}" for t in yrt]
fig.add_scatter(x=[xmin], y=[ymin], xaxis='x2', mode='markers', marker=dict(opacity=0), showlegend=False, hoverinfo='skip')
fig.update_layout(template='plotly_white', height=560, width=980, margin=dict(t=120, r=40, b=60, l=72),
                  font=dict(family='Helvetica Neue, Arial, sans-serif', size=13, color='#222'),
                  title=dict(text='SoH Prediction for Resold Vehicles', x=0.02, xanchor='left', y=0.97, yref='container', font=dict(size=18)),
                  xaxis=dict(title='calendar', type='date', range=[xmin, xmax], showgrid=True, gridcolor='rgba(0,0,0,0.06)', ticks='outside'),
                  xaxis2=dict(overlaying='x', side='top', type='date', range=[xmin, xmax], tickmode='array', tickvals=yrt, ticktext=age_txt,
                              title=dict(text='age since registration (years)', standoff=8), showgrid=False, ticks='outside', tickfont=dict(size=10)),
                  yaxis=dict(title='State of Health (%)', range=[ymin, 101], showgrid=True, gridcolor='rgba(0,0,0,0.06)', ticks='outside'),
                  legend=dict(title=dict(text='Vehicle · SoH at resale', font=dict(size=11)), x=1.01, y=1, font=dict(size=11), bgcolor='rgba(0,0,0,0)'))
fig.show()